# Synthetic Image Parameters

This notebook explores how `render_echelle_lines` parameters change the
appearance of the synthetic detector frame.

| Parameter | Physical / instrumental? |
|---|---|
| `psf_sigma_px` | Physical — set by optics + pixel sampling |
| `order_spacing_px` | Instrumental — depends on prism cross-disperser angle and geometry |
| `background` | Instrumental — detector dark current, sky |
| `read_noise` | Instrumental — detector read noise |

The x-position of lines (echelle dispersion) is always computed from the
grating equation and is **not** a free parameter.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from echelle_optics import lhd_cmos_echelle, render_echelle_lines

spec   = lhd_cmos_echelle()
orders = list(range(33, 54))   # subset: keeps images compact
SHAPE  = (420, 700)

lines = [
    (404.656, 0.6),
    (435.833, 0.7),
    (486.133, 0.8),
    (546.074, 1.0),
    (587.562, 0.9),
    (656.281, 1.0),
]

def show(img, title, cmap="gray"):
    fig, ax = plt.subplots(figsize=(9, 4))
    vmax = np.percentile(img if img.ndim == 2 else img.max(-1), 99)
    kw = dict(origin="upper", aspect="auto")
    if img.ndim == 2:
        ax.imshow(img, cmap=cmap, vmin=0, vmax=max(vmax, 1e-6), **kw)
    else:
        ax.imshow((img / max(img.max(), 1e-6)).clip(0, 1), **kw)
    ax.set_title(title)
    ax.set_xlabel("Dispersion (px)")
    ax.set_ylabel("Cross-disp. (px)")
    plt.tight_layout()
    plt.show()

## Varying PSF size

`psf_sigma_px` controls the Gaussian line spread on the detector.
Typical seeing-limited spectrographs have $\sigma \approx 1$–3 px.

In [2]:
for sigma in [0.7, 1.5, 3.5]:
    img = render_echelle_lines(
        lines, spec, orders=orders, shape=SHAPE,
        order_spacing_px=19.0, psf_sigma_px=sigma,
        background=0.0, read_noise=0.0,
    )
    show(img, f"psf_sigma_px = {sigma}")

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

## Varying order spacing

`order_spacing_px` is the cross-dispersion separation between adjacent orders
in pixels.  It encodes the prism geometry and cannot currently be derived from
first principles in this package — it must be calibrated from the detector.

In [3]:
for spacing in [10.0, 20.0, 35.0]:
    img = render_echelle_lines(
        lines, spec, orders=orders, shape=SHAPE,
        order_spacing_px=spacing, psf_sigma_px=1.5,
        background=0.0, read_noise=0.0,
    )
    show(img, f"order_spacing_px = {spacing}")

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

## Background and read noise

`background` adds a uniform offset (dark current, sky).  
`read_noise` adds Gaussian noise drawn once per pixel (fixed with `seed`).

In [4]:
base_kw = dict(
    orders=orders, shape=SHAPE,
    order_spacing_px=19.0, psf_sigma_px=1.5,
)

# No noise baseline
img_clean = render_echelle_lines(lines, spec, background=0.0, read_noise=0.0, **base_kw)
show(img_clean, "Clean: background=0, read_noise=0")

# Background only
img_bg = render_echelle_lines(lines, spec, background=0.05, read_noise=0.0, **base_kw)
show(img_bg, "background=0.05, read_noise=0")

# Read noise only
img_rn = render_echelle_lines(lines, spec, background=0.0, read_noise=0.05, seed=7, **base_kw)
show(img_rn, "background=0, read_noise=0.05")

# Both
img_both = render_echelle_lines(lines, spec, background=0.05, read_noise=0.05, seed=7, **base_kw)
show(img_both, "background=0.05, read_noise=0.05")

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

## Colour rendering for the same scene

Colour assigns approximate visible-spectrum hues to lines but does not
change the x/y positions — it is purely cosmetic.

In [5]:
img_color = render_echelle_lines(
    lines, spec, color=True,
    background=0.0, read_noise=0.0,
    **base_kw,
)
show(img_color, "Colour rendering")

<Figure size 900x400 with 1 Axes>